<a href="https://colab.research.google.com/github/Razae786/NLP-Lab-FA23-BAI-064/blob/main/NLP_Lab7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **FA23-BAI-064**

# Lab 7: POS Tagging & NER (AIC365)
**Data note:** the instructor's news paragraph and job postings were not available, so this notebook uses SAMPLE data in the two data cells (marked `# DATA`). To use the real data, replace only those two variables, then Run All.

In [1]:
import spacy, pandas as pd
from spacy import displacy
from collections import Counter, defaultdict
nlp = spacy.load("en_core_web_sm")

# Stage a1: News paragraph

In [2]:
# DATA: replace with your real paragraph if you have it
text = ("Google will launch the Pixel 10 next month. The launch event is in New York on "
        "August 20, 2025, and the phone would ship to over 40 countries at $799. "
        "Apple said iPhone sales fell 5%, said Tim Cook.")
doc = nlp(text)

## Activity 1: POS (coarse `.pos_`) and TAG (fine `.tag_`)
`.pos_` is the Universal Dependencies tag; `.tag_` is the Penn Treebank tag (adds tense/number). Unexpected tag: **"ship"** is normally a noun, but here it is a VERB because "would ship to" puts it after a modal. The tagger reads neighbouring tokens (tok2vec context window), not the word alone.

In [3]:
print(f"{'Text':<15} | {'POS':<8} | TAG")
print("-" * 35)
for t in doc:
    print(f"{t.text:<15} | {t.pos_:<8} | {t.tag_}")

Text            | POS      | TAG
-----------------------------------
Google          | PROPN    | NNP
will            | AUX      | MD
launch          | VERB     | VB
the             | DET      | DT
Pixel           | PROPN    | NNP
10              | NUM      | CD
next            | ADJ      | JJ
month           | NOUN     | NN
.               | PUNCT    | .
The             | DET      | DT
launch          | NOUN     | NN
event           | NOUN     | NN
is              | AUX      | VBZ
in              | ADP      | IN
New             | PROPN    | NNP
York            | PROPN    | NNP
on              | ADP      | IN
August          | PROPN    | NNP
20              | NUM      | CD
,               | PUNCT    | ,
2025            | NUM      | CD
,               | PUNCT    | ,
and             | CCONJ    | CC
the             | DET      | DT
phone           | NOUN     | NN
would           | AUX      | MD
ship            | VERB     | VB
to              | ADP      | IN
over            | ADP      | IN


## Activity 2: same word, two roles
"launch" is a VERB in "will launch the Pixel" and a NOUN in "launch event". A dictionary lookup cannot do this; sequence labeling uses context.

In [4]:
target_word = "launch"
for t in doc:
    if t.lower_ == target_word:
        print(f"...{doc[max(0, t.i-2): t.i+3].text}... | {t.text} | {t.pos_}")

...Google will launch the Pixel... | launch | VERB
.... The launch event is... | launch | NOUN


## Activity 3: remove noise tokens
Only the printout is filtered; `doc` still contains punctuation and spaces.

In [5]:
noise = {"SPACE", "PUNCT", "X"}
print(f"{'Clean Text':<15} | POS")
print("-" * 25)
for t in doc:
    if t.pos_ not in noise:
        print(f"{t.text:<15} | {t.pos_}")

Clean Text      | POS
-------------------------
Google          | PROPN
will            | AUX
launch          | VERB
the             | DET
Pixel           | PROPN
10              | NUM
next            | ADJ
month           | NOUN
The             | DET
launch          | NOUN
event           | NOUN
is              | AUX
in              | ADP
New             | PROPN
York            | PROPN
on              | ADP
August          | PROPN
20              | NUM
2025            | NUM
and             | CCONJ
the             | DET
phone           | NOUN
would           | AUX
ship            | VERB
to              | ADP
over            | ADP
40              | NUM
countries       | NOUN
at              | ADP
$               | SYM
799             | NUM
Apple           | PROPN
said            | VERB
iPhone          | PROPN
sales           | NOUN
fell            | VERB
5               | NUM
%               | NOUN
said            | VERB
Tim             | PROPN
Cook            | PROPN


## Activity 4: NER, displaCy, counts
`doc.ents` holds Span objects (BILOU boundaries already decoded into spans).

In [6]:
displacy.render(doc, style="ent", jupyter=True)
counts = Counter(e.label_ for e in doc.ents)
for label in ["PERSON", "ORG", "GPE", "DATE", "MONEY", "PERCENT"]:
    print(label, counts.get(label, 0))
print("All types:", dict(counts))

PERSON 2
ORG 2
GPE 1
DATE 2
MONEY 1
PERCENT 1
All types: {'ORG': 2, 'PERSON': 2, 'DATE': 2, 'GPE': 1, 'CARDINAL': 1, 'MONEY': 1, 'PERCENT': 1}


## Activity 5: fix a missed entity with EntityRuler
The small model mislabels or misses the product names (see BEFORE). The ruler is placed `before="ner"`, so its spans win on overlap. Matching is case-sensitive.

In [7]:
print("BEFORE:", [(e.text, e.label_) for e in doc.ents])

nlp_custom = spacy.load("en_core_web_sm")
ruler = nlp_custom.add_pipe("entity_ruler", before="ner")
ruler.add_patterns([{"label": "PRODUCT", "pattern": "Pixel 10"},
                    {"label": "PRODUCT", "pattern": "iPhone"}])
doc2 = nlp_custom(text)
print("AFTER :", [(e.text, e.label_) for e in doc2.ents])

BEFORE: [('Google', 'ORG'), ('Pixel', 'PERSON'), ('next month', 'DATE'), ('New York', 'GPE'), ('August 20, 2025', 'DATE'), ('40', 'CARDINAL'), ('799', 'MONEY'), ('Apple', 'ORG'), ('5%', 'PERCENT'), ('Tim Cook', 'PERSON')]
AFTER : [('Google', 'ORG'), ('Pixel 10', 'PRODUCT'), ('next month', 'DATE'), ('New York', 'GPE'), ('August 20, 2025', 'DATE'), ('40', 'CARDINAL'), ('799', 'MONEY'), ('Apple', 'ORG'), ('iPhone', 'PRODUCT'), ('5%', 'PERCENT'), ('Tim Cook', 'PERSON')]


# Stage v: Job postings

In [8]:
# DATA: replace with your real postings (list of strings) if you have them
postings = [
    "Google is hiring a Python developer in Lahore. Experience with AWS and Figma is required.",
    "Systems Ltd in Karachi needs engineers to design APIs using Python and Docker. Strong SQL skills required.",
    "Microsoft seeks a React developer in Islamabad to build and test web applications. Knowledge of AWS preferred.",
    "Amazon is looking for a data analyst in Lahore. You will support the team with Python, SQL and Figma dashboards.",
    "Netsol Technologies in Lahore hires a UI designer. Design experience in Figma and customer support skills needed.",
    "Careem needs a backend engineer in Karachi to test and launch new services. Docker and AWS experience required.",
]
docs = list(nlp.pipe(postings))

## Activity 1: POS table, top 5 nouns and verbs
Counted by `lemma_` ("skill" and "skills" merge); `is_alpha` drops numbers and symbols; "is/be" are AUX so they stay out of verbs.

In [9]:
rows = [(i, t.text, t.pos_, t.tag_) for i, d in enumerate(docs, 1) for t in d if not t.is_space]
pos_df = pd.DataFrame(rows, columns=["posting", "token", "pos", "tag"])
print(pos_df.head(20))

def top5(pos):
    return Counter(t.lemma_.lower() for d in docs for t in d if t.pos_ == pos and t.is_alpha).most_common(5)

print("Top 5 nouns:", top5("NOUN"))
print("Top 5 verbs:", top5("VERB"))

    posting       token    pos   tag
0         1      Google  PROPN   NNP
1         1          is    AUX   VBZ
2         1      hiring   VERB   VBG
3         1           a    DET    DT
4         1      Python  PROPN   NNP
5         1   developer   NOUN    NN
6         1          in    ADP    IN
7         1      Lahore  PROPN   NNP
8         1           .  PUNCT     .
9         1  Experience   NOUN    NN
10        1        with    ADP    IN
11        1         AWS  PROPN  NNPS
12        1         and  CCONJ    CC
13        1       Figma  PROPN   NNP
14        1          is    AUX   VBZ
15        1    required   VERB   VBN
16        1           .  PUNCT     .
17        2     Systems  PROPN  NNPS
18        2         Ltd  PROPN   NNP
19        2          in    ADP    IN
Top 5 nouns: [('experience', 3), ('developer', 2), ('engineer', 2), ('skill', 2), ('api', 1)]
Top 5 verbs: [('require', 3), ('need', 3), ('hire', 2), ('design', 1), ('use', 1)]


## Activity 2: NER counts per type
The default small model often labels skills (Python, AWS) as ORG or PERSON, which inflates those counts. Write your explanation of the top type from the printed output below. Postings usually name companies and cities, so ORG and GPE are the expected leaders.

In [10]:
ent_counts = Counter(e.label_ for d in docs for e in d.ents)
print(ent_counts.most_common())
groups = defaultdict(list)
for d in docs:
    for e in d.ents:
        groups[e.label_].append(e.text)
for k, v in groups.items():
    print(k, "->", v)

[('ORG', 11), ('GPE', 7), ('PERSON', 4)]
ORG -> ['Google', 'AWS', 'Figma', 'SQL', 'Microsoft', 'AWS', 'Amazon', 'SQL', 'Figma', 'UI', 'Figma']
GPE -> ['Lahore', 'Karachi', 'React', 'Islamabad', 'Lahore', 'Lahore', 'Karachi']
PERSON -> ['Python', 'Docker', 'Netsol Technologies', 'Docker']


## Activity 3: EntityRuler for skills
Skill terms are missed or mislabeled by the default model. The ruler runs before `ner` and `phrase_matcher_attr="LOWER"` makes it case-insensitive.

In [11]:
nlp2 = spacy.load("en_core_web_sm")
ruler = nlp2.add_pipe("entity_ruler", before="ner", config={"phrase_matcher_attr": "LOWER"})
skills = ["Python", "AWS", "Figma", "Docker", "SQL", "React"]
ruler.add_patterns([{"label": "SKILL", "pattern": s} for s in skills])

docs2 = list(nlp2.pipe(postings))
for i, d in enumerate(docs2, 1):
    print(i, [e.text for e in d.ents if e.label_ == "SKILL"])

1 ['Python', 'AWS', 'Figma']
2 ['Python', 'Docker', 'SQL']
3 ['React', 'AWS']
4 ['Python', 'SQL', 'Figma']
5 ['Figma']
6 ['Docker', 'AWS']


## Activity 4: one word, different POS across postings
Prints any word seen as both NOUN and VERB, with context.

In [12]:
seen = defaultdict(dict)
for i, d in enumerate(docs, 1):
    for t in d:
        if t.pos_ in ("NOUN", "VERB") and t.is_alpha:
            seen[t.lower_].setdefault(t.pos_, (i, t.tag_, d[max(0, t.i-3): t.i+4].text))
for word, v in seen.items():
    if "NOUN" in v and "VERB" in v:
        print(f"\n'{word}'")
        for pos, (i, tag, ctx) in v.items():
            print(f"  posting {i} | {pos}/{tag} | ...{ctx}...")


'design'
  posting 2 | VERB/VB | ...needs engineers to design APIs using Python...
  posting 5 | NOUN/NN | ...UI designer. Design experience in Figma...

'test'
  posting 3 | NOUN/NN | ...to build and test web applications....
  posting 6 | VERB/VB | ...in Karachi to test and launch new...

'support'
  posting 4 | VERB/VB | .... You will support the team with...
  posting 5 | NOUN/NN | ...Figma and customer support skills needed....


## Activity 5: structured extractor
Uses `docs2` (ruler pipeline) so the SKILL label exists. `dict.fromkeys` removes duplicates in order.

In [13]:
def uniq(d, label):
    return ", ".join(dict.fromkeys(e.text for e in d.ents if e.label_ == label))

df = pd.DataFrame([{"posting_id": i, "company": uniq(d, "ORG"), "location": uniq(d, "GPE"),
                    "skills": uniq(d, "SKILL")} for i, d in enumerate(docs2, 1)])
print(df.to_string(index=False))
df.to_csv("postings_structured.csv", index=False)

 posting_id   company  location              skills
          1    Google    Lahore  Python, AWS, Figma
          2             Karachi Python, Docker, SQL
          3 Microsoft Islamabad          React, AWS
          4    Amazon    Lahore  Python, SQL, Figma
          5        UI    Lahore               Figma
          6             Karachi         Docker, AWS
